# Painting the Obama head

Paints the scan from seven angles, plus a green floral backdrop, so the paintings can be wrapped back onto the 3D head, and the cube joiner then photographs the painted head.

Charcoal drawing with bright, saturated "presidential" color, after the spirit of Kehinde Wiley's official portrait (vivid greens, royal blue, gold, rich skin tones). The style is described in the prompt, not by naming the artist.

1. **Runtime → Change runtime type → GPU** (a T4 works; an A100 is faster).
2. Run the cells in order. The guides load from [`obama_kit/`](obama_kit) in the repo, so there's nothing to upload.
3. The front view is painted first. Every other view uses it as a style reference, so the views agree.
4. Then the floral backdrop is painted, to sit behind the whole model.
5. Download `paintings.zip` at the end; the joiner wraps the paintings back onto the head.

The head is "BARACK OBAMA" by louis (https://sketchfab.com/louis), from https://sketchfab.com/3d-models/barack-obama-3010d6b0fdd843b397fa9e98437bc22a, licensed CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). We cleaned and cropped it and rendered the guides from it.

In [ ]:
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1

In [ ]:
# The guides (photo, depth and mask for each view, plus the cameras) come straight from the repo.
import json, os, urllib.request
BRANCH = "cubism/obama"   # "main" once merged
BASE = f"https://raw.githubusercontent.com/rath-hy/folded-illusions/{BRANCH}/cubism/obama_kit/"
os.makedirs("kit", exist_ok=True)
urllib.request.urlretrieve(BASE + "cameras.json", "kit/cameras.json")
CAMS = json.load(open("kit/cameras.json"))
VIEWS = list(CAMS["views"])
for name in VIEWS:
    for kind in ("photo", "depth", "mask"):
        urllib.request.urlretrieve(BASE + f"{name}_{kind}.png", f"kit/{name}_{kind}.png")
print(VIEWS)

In [ ]:
import torch
from diffusers import ControlNetModel, AutoencoderKL, StableDiffusionXLControlNetImg2ImgPipeline
controlnet = ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16)
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=controlnet, vae=vae,
    torch_dtype=torch.float16, variant="fp16").to("cuda")
try:   # style reference: the front painting guides the rest
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
    HAS_IP = True
except Exception as e:
    print("No IP-Adapter, views will be matched by prompt and seed only:", e)
    HAS_IP = False

## Settings
- **STRENGTH**: how far the paint departs from the photo (0.5 keeps him very close; 0.8 is loose).
- **CONTROL**: how tightly the depth map holds the shape.
- **STYLE**: how strongly the front painting steers the other views (IP-Adapter).

In [ ]:
PROMPT = ("expressive charcoal drawing portrait of Barack Obama, bold charcoal strokes, smudges and hatching on paper, "
          "the planes of the face subtly faceted with a few angular, geometric strokes, still a clear and natural likeness, "
          "bright saturated presidential colors woven into the charcoal: vivid emerald green, royal blue and gold accents, "
          "rich warm brown skin tones, crisp realist contemporary American portraiture, confident, dignified, highly detailed face")
NEGATIVE = ("photo, 3d render, plastic, blurry, deformed, extra eyes, extra nose, text, signature, watermark, frame, "
            "dull, desaturated, monochrome only, background scenery, heavily abstract, shattered, fragmented, mosaic")
# The backdrop: lush green foliage and flowers (chrysanthemums for Chicago, jasmine for Hawaii, blue lilies for
# Kenya), painted loosely, not geometric. It sits behind the whole model.
BACKDROP_PROMPT = ("dense lush wall of vivid green foliage and leaves with scattered flowers, white jasmine, "
                   "purple and white chrysanthemums, blue lilies, bright saturated color, painterly with charcoal "
                   "accents, flat frontal view, edge to edge, no people, no sky")
BACKDROP_NEGATIVE = "person, face, text, signature, watermark, frame, geometric, cubist, dull, desaturated, sky, horizon"
STRENGTH, CONTROL, STYLE = 0.62, 0.6, 0.5
STEPS, GUIDANCE, SEED = 35, 6.0, 7


In [ ]:
from PIL import Image, ImageFilter
import numpy as np
os.makedirs("paintings", exist_ok=True)

def paint(name, seed=SEED, ref=None):
    photo = Image.open(f"kit/{name}_photo.png").convert("RGB")
    depth = Image.open(f"kit/{name}_depth.png").convert("RGB")
    mask = Image.open(f"kit/{name}_mask.png").convert("L").filter(ImageFilter.MaxFilter(9))
    kw = {}
    if HAS_IP:
        pipe.set_ip_adapter_scale(STYLE if ref is not None else 0.0)
        kw["ip_adapter_image"] = ref if ref is not None else photo
    out = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=photo, control_image=depth,
               strength=STRENGTH, controlnet_conditioning_scale=CONTROL, num_inference_steps=STEPS,
               guidance_scale=GUIDANCE, generator=torch.Generator("cuda").manual_seed(seed), **kw).images[0]
    out = Image.composite(out, Image.new("RGB", out.size, "white"), mask)   # only the head matters
    out.save(f"paintings/{name}.png")
    return out

front = paint("front")
front

If the front isn't right, change the settings or `SEED` and rerun the cell above. Everything else follows from it.

In [ ]:
results = {"front": front}
for name in VIEWS:
    if name != "front":
        results[name] = paint(name, ref=front)
grid = Image.new("RGB", (256 * len(results), 256), "white")
for i, im in enumerate(results.values()):
    grid.paste(im.resize((256, 256)), (256 * i, 0))
grid

## The floral backdrop
One painting behind the whole model. Wherever a tile shows no head, it shows this, so head-on it reads as one continuous background. The front painting steers its palette lightly.

In [ ]:
from diffusers import StableDiffusionXLPipeline
t2i = StableDiffusionXLPipeline(**{k: v for k, v in pipe.components.items() if k in
    ("vae", "text_encoder", "text_encoder_2", "tokenizer", "tokenizer_2", "unet", "scheduler",
     "image_encoder", "feature_extractor")})
kw = {}
if HAS_IP:
    t2i.set_ip_adapter_scale(0.25)
    kw["ip_adapter_image"] = front
backdrop = t2i(prompt=BACKDROP_PROMPT, negative_prompt=BACKDROP_NEGATIVE, width=1024, height=1024,
               num_inference_steps=STEPS, guidance_scale=GUIDANCE,
               generator=torch.Generator("cuda").manual_seed(SEED), **kw).images[0]
backdrop.save("paintings/backdrop.png")
backdrop

### Redo one view

In [ ]:
# results["profile"] = paint("profile", seed=11, ref=front); results["profile"]

### Optional: sharpen up
Close-up tiles enlarge a small part of a painting, so this repaints each view at 2x with a light touch. It's slower.

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
UPSCALE = False
if UPSCALE:  # (the backdrop is left as is; it's soft by nature)
    up_pipe = StableDiffusionXLImg2ImgPipeline(**{k: v for k, v in pipe.components.items() if k in
        ("vae", "text_encoder", "text_encoder_2", "tokenizer", "tokenizer_2", "unet", "scheduler")})
    if HAS_IP:
        pipe.unload_ip_adapter()
    for name in VIEWS:
        im = Image.open(f"paintings/{name}.png").resize((2048, 2048), Image.LANCZOS)
        big = up_pipe(prompt=PROMPT, negative_prompt=NEGATIVE, image=im, strength=0.25, num_inference_steps=30,
                      guidance_scale=5.0, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
        mask = Image.open(f"kit/{name}_mask.png").convert("L").filter(ImageFilter.MaxFilter(9)).resize((2048, 2048))
        Image.composite(big, Image.new("RGB", big.size, "white"), mask).save(f"paintings/{name}.png")

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("paintings", "zip", "paintings")
files.download("paintings.zip")